<a href="https://colab.research.google.com/github/JiaxunBao/MATH509-Final-Project/blob/JB-branch/JBbranch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import Libraries

In [8]:
import pandas as pd
import numpy as np

## Read data

In [9]:
from google.colab import drive
drive.mount('/content/drive')

FILE_PATH = "/content/drive/MyDrive/MATH509finalproject/Data1(MATH509).xlsx"
SHEET_NAME = 0
OBSERVED_CUTOFF = "2024Q2"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Load the first sheet

In [10]:
def q_to_period(q):
    return pd.Period(str(q).strip().upper(), freq="Q")

df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
df.columns = df.columns.str.strip()

## Sort and create lagged predictors within each CMA

In [11]:
df["quarter_period"] = df["quarter"].apply(q_to_period)
df = df.sort_values(["cma", "quarter_period"]).copy()

base_macro = [
    "inflation",
    "disposable_income",
    "total_debt_payments",
    "mortgage_interest_paid",
    "bank_rate",
]

df["delinq_lag1"] = df.groupby("cma")["delinq_index_2012Q3_100"].shift(1)
for col in base_macro:
    df[f"{col}_lag1"] = df.groupby("cma")[col].shift(1)

features = ["delinq_lag1"] + [f"{c}_lag1" for c in base_macro]

## Keep observed rows only for model selection

In [12]:
observed_cutoff = pd.Period(OBSERVED_CUTOFF, freq="Q")
obs_df = df[df["quarter_period"] <= observed_cutoff].dropna(
    subset=["delinq_index_2012Q3_100"] + features
).copy()

# Optional future rows in the first sheet
future_df = df[df["quarter_period"] > observed_cutoff].dropna(subset=features).copy()

## 4. Time-based holdout: last 8 observed quarters

In [13]:
obs_quarters = sorted(obs_df["quarter_period"].unique())
test_quarters = obs_quarters[-8:]

train_df = obs_df[~obs_df["quarter_period"].isin(test_quarters)].copy()
test_df = obs_df[obs_df["quarter_period"].isin(test_quarters)].copy()

X_train = train_df[["cma"] + features]
y_train = train_df["delinq_index_2012Q3_100"]
X_test = test_df[["cma"] + features]
y_test = test_df["delinq_index_2012Q3_100"]